In [0]:
from pyspark.sql import functions as F

FEATURE_TABLE = "oil_gas_demo.gold.demand_features"
AGG_TABLE = "oil_gas_demo.gold.daily_demand_agg"


features_df = spark.table(FEATURE_TABLE)
daily_df = spark.table(AGG_TABLE)

print("Gold tables loaded successfully!")

In [0]:

print("Demand features rows:", features_df.count())
print("Daily demand rows:", daily_df.count())


print("\nDemand Features Schema:")
features_df.printSchema()

print("\nDaily Demand Schema:")
daily_df.printSchema()



In [0]:
duplicate_rows = (
    daily_df
    .groupBy(
        "transaction_date",
        "product_name",
        "destination_city"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate date-product-city combinations:")


In [0]:
from pyspark.sql import functions as F

# Load actual daily demand
actual_demand = (
    spark.table("oil_gas_demo.gold.daily_demand_agg")
    .select(
        F.to_date("transaction_date").alias("actual_date"),
        "product_name",
        "destination_city",
        F.col("total_demand").alias("target_demand")
    )
)

display(actual_demand.limit(10))

In [0]:

features_df = spark.table(
    "oil_gas_demo.gold.demand_features"
)

features_with_target_date = features_df.withColumn(
    "target_date",
    F.date_add(
        F.to_date("transaction_date"),
        7
    )
)

display(
    features_with_target_date.select(
        "transaction_date",
        "target_date",
        "product_name",
        "destination_city",
        "total_demand"
    ).limit(10)
)

In [0]:
# Match each forecast date to actual demand 7 days later
target_dataset = (
    features_with_target_date.alias("f")
    .join(
        actual_demand.alias("a"),
        (
            (F.col("f.target_date") == F.col("a.actual_date")) &
            (F.col("f.product_name") == F.col("a.product_name")) &
            (F.col("f.destination_city") == F.col("a.destination_city"))
        ),
        "left"
    )
    .select(
        F.col("f.transaction_date").alias("forecast_date"),
        F.col("f.target_date"),
        F.col("f.product_name"),
        F.col("f.destination_city"),
        F.col("f.total_demand"),
        F.col("f.avg_unit_price"),
        F.col("f.total_inventory"),
        F.col("f.day_of_week"),
        F.col("f.month"),
        F.col("f.demand_lag_1"),
        F.col("f.demand_lag_7"),
        F.col("f.rolling_avg_7"),
        F.col("f.rolling_std_7"),
        F.col("a.target_demand")
    )
)

display(target_dataset.limit(20))

In [0]:
# Count matched and unmatched target rows
target_dataset.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.col("target_demand").isNotNull(), 1)
         .otherwise(0)
    ).alias("matched_rows"),
    F.sum(
        F.when(F.col("target_demand").isNull(), 1)
         .otherwise(0)
    ).alias("unmatched_rows")
).show()

In [0]:
# Keep only rows with a known actual target
training_dataset = target_dataset.filter(
    F.col("target_demand").isNotNull()
)

print("Rows available for modeling:", training_dataset.count())

display(training_dataset.limit(10))    # it is used for modeling

In [0]:
from pyspark.sql import functions as F

KEYS = [
    "forecast_date",
    "product_name",
    "destination_city"
]

duplicate_keys = (
    training_dataset
    .groupBy(*KEYS)
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate keys:", duplicate_keys.count())
display(duplicate_keys.limit(10))

In [0]:
import importlib, databricks
importlib.reload(databricks)

from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()


In [0]:
FEATURE_TABLE = "oil_gas_demo.gold.demand_training_features"

fe.create_table(
    name=FEATURE_TABLE,
    primary_keys=KEYS,
    df=training_dataset,
    description=(
        "Prepared demand forecasting features with actual demand "
        "7 calendar days ahead as the training target."
    ),
    tags={
        "project": "oil_gas_demand_forecasting",
        "model": "random_forest",
        "forecast_horizon": "7_days"
    }
)

print("Feature table ", FEATURE_TABLE)

In [0]:
saved_features = fe.read_table(
    name=FEATURE_TABLE
)

print("Saved rows:", saved_features.count())

display(
    saved_features
    .orderBy("forecast_date")
    .limit(10)
)